<a href="https://colab.research.google.com/github/arvinmungara-alt/Agentic-AI/blob/main/AI_project_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q -U langchain langchain-community langchain-text-splitters chromadb sentence-transformers pypdf transformers

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = PyPDFLoader("Public_Health_Protocol.pdf")
pages = loader.load_and_split()

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
chunks = splitter.split_documents(pages)

print(f"Loaded {len(pages)} pages, split into {len(chunks)} chunks")

Loaded 5 pages, split into 9 chunks


In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:
from langchain_community.vectorstores import Chroma

db = Chroma.from_documents(chunks, embeddings)
print("Database built with", db._collection.count(), "chunks")

Database built with 13 chunks


In [ ]:
retriever = db.as_retriever()

In [ ]:
from langchain_community.llms import HuggingFacePipeline
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch
from langchain_core.language_models.llms import LLM
from typing import Optional, List

model_name = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

class FlanT5LLM(LLM):
    @property
    def _llm_type(self) -> str:
        return "flan-t5"

    def _call(self, prompt: str, stop: Optional[List[str]] = None) -> str:
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024)
        outputs = model.generate(**inputs, max_new_tokens=256)
        return tokenizer.decode(outputs[0], skip_special_tokens=True)

llm = FlanT5LLM()

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

prompt_template = """Use the following context to answer the question.
If the answer is not contained in the context, say "I cannot answer this based on the provided documents."

Context: {context}

Question: {question}
Answer:"""

PROMPT = PromptTemplate(template=prompt_template, input_variables=["context", "question"])

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

def qa_chain_invoke(query):
    docs = retriever.invoke(query)
    context = format_docs(docs)
    prompt_text = PROMPT.format(context=context, question=query)
    answer = llm.invoke(prompt_text)
    return {"result": answer, "source_documents": docs}

In [ ]:
result = qa_chain_invoke("How quickly must Category A Virus cases be reported?")

print("ANSWER:")
print(result["result"])
print("\nSOURCE CHUNKS USED:")
for doc in result["source_documents"]:
    print("-", doc.page_content[:150], "...")


result = qa_chain_invoke("What PPE is required?")

print("\n\nANSWER:")
print(result["result"])
print("\nSOURCE CHUNKS USED:")
for doc in result["source_documents"]:
    print("-", doc.page_content[:150], "...")


result = qa_chain_invoke("How long are incident reports retained?")

print("\n\nANSWER:")
print(result["result"])
print("\nSOURCE CHUNKS USED:")
for doc in result["source_documents"]:
    print("-", doc.page_content[:150], "...")


result = qa_chain_invoke("Who can talk to the media?")

print("\n\nANSWER:")
print(result["result"])
print("\nSOURCE CHUNKS USED:")
for doc in result["source_documents"]:
    print("-", doc.page_content[:150], "...")


result = qa_chain_invoke("Can PHI be uploaded to public AI tools?")

print("\n\nANSWER:")
print(result["result"])
print("\nSOURCE CHUNKS USED:")
for doc in result["source_documents"]:
    print("-", doc.page_content[:150], "...")


result = qa_chain_invoke("What triggers notification of the Regional Response Center?")

print("\n\nANSWER:")
print(result["result"])
print("\nSOURCE CHUNKS USED:")
for doc in result["source_documents"]:
    print("-", doc.page_content[:150], "...")

ANSWER:
within 2 hours

SOURCE CHUNKS USED:
- Category  A  virus  Rporting  protocol:  use  the  in  house  virus  firma  medicine  and  report  to  
maryland
 
hospital ...
- Category  A  virus  Rporting  protocol:  use  the  in  house  virus  firma  medicine  and  report  to  
maryland
 
hospital ...
- appropriate
 
protective
 
measures,
 
and
 
secure
 
handling
 
of
 
patient
 
information.
  ------------------------------------------------------- ...
- Healthcare  facilities  must  retain  records  according  to  the  following  schedule:   Incident  Reports:  Retention  Period:  7  years   Laborator ...


ANSWER:
N95 respirators

SOURCE CHUNKS USED:
- Healthcare  facilities  must  retain  records  according  to  the  following  schedule:   Incident  Reports:  Retention  Period:  7  years   Laborator ...
- N95  respirators  are  required  for  direct  patient  care  involving  suspected  Category  A  Virus.   Policy  PH-303:  Laboratory  specimens  must  ...
- appropriate
 
prote